## Задача 1

По минимальным многочленам $M_1(x)$ и $M_3(x)$:
- построить порождающий полином $g(x)$,
- построить порождающую матрицу $G$ в двоичном виде,
- проверить, что $G \cdot H^T = 0$.


In [9]:
def poly_mul_bin(p, q):  # Умножение бинарных полиномов (коэффициенты low->high)
    res = [0] * (len(p) + len(q) - 1)  # создаем массив для хранения результата
    for i, pi in enumerate(p):  # проходим по всем коэффициентам первого полинома
        if pi:  # если коэффициент ненулевой
            for j, qj in enumerate(
                q
            ):  # проходим по всем коэффициентам второго полинома
                if qj:  # если коэффициент второго полинома ненулевой
                    res[i + j] ^= 1  # складываем по модулю 2
    return res


def poly_div_rem_bin(
    dividend, divisor
):  # Деление бинарных полиномов: возвращает (частное, остаток)
    tmp = dividend[:]  # создаем копию делимого
    q = [0] * (len(dividend) - len(divisor) + 1)  # создаем массив для хранения частного
    deg_d = len(divisor) - 1  # вычисляем степень делителя
    for i in range(
        len(tmp) - 1, deg_d - 1, -1
    ):  # проходим по всем коэффициентам делимого
        if tmp[i]:  # если коэффициент ненулевой
            qi = i - deg_d  # вычисляем индекс для записи в частное
            q[qi] = 1  # записываем 1 в частное
            for j, dj in enumerate(divisor):  # проходим по всем коэффициентам делителя
                if dj:  # если коэффициент делителя ненулевой
                    tmp[qi + j] ^= 1  # складываем по модулю 2
    rem = tmp[:deg_d]  # вычисляем остаток
    return q, rem  # возвращаем частное и остаток


def poly_to_str(poly):  # Преобразование полинома в строку
    terms = []
    for i, c in enumerate(poly):  # проходим по всем коэффициентам полинома
        if c:  # если коэффициент ненулевой
            if i == 0:  # если коэффициент при x^0
                terms.append("1")  # добавляем "1" в список термов
            elif i == 1:  # если коэффициент при x^1
                terms.append("x")  # добавляем "x" в список термов
            else:  # если коэффициент при x^i
                terms.append(f"x^{i}")  # добавляем "x^i" в список термов
    return (
        " + ".join(terms) if terms else "0"
    )  # возвращаем строку, содержащую все термы полинома


def cyclic_generator_matrix(g, n, k):  # Стандартная циклическая форма G: k сдвигов g(x)
    rows = []  # создаем список для хранения строк порождающей матрицы
    for i in range(k):  # проходим по всем строкам порождающей матрицы
        row = [0] * i + g + [0] * (n - len(g) - i)  # создаем строку порождающей матрицы
        rows.append(row)  # добавляем строку в список строк порождающей матрицы
    return rows  # возвращаем список строк порождающей матрицы


def cyclic_parity_check_matrix(
    h, n, r
):  #    Стандартная циклическая форма H: r сдвигов h(x),  где r=n-k=deg(g), deg(h)=k
    rows = []
    for i in range(r):  # проходим по всем строкам проверочной матрицы
        row = [0] * i + h + [0] * (n - len(h) - i)  # создаем строку проверочной матрицы
        rows.append(row)  # добавляем строку в список строк проверочной матрицы
    return rows  # возвращаем список строк проверочной матрицы


def mat_mul_mod2(A, B):  # Умножение матриц по модулю 2
    rows = len(A)  # количество строк в матрице A
    cols = len(B[0])  # количество столбцов в матрице B
    mid = len(B)  # количество строк в матрице B
    C = [
        [0] * cols for _ in range(rows)
    ]  # создаем матрицу C с количеством строк равным количеству строк в матрице A и количеством столбцов равным количеству столбцов в матрице B
    for i in range(rows):  # проходим по всем строкам матрицы A
        for j in range(cols):  # проходим по всем столбцам матрицы B
            s = 0  # инициализируем переменную s для хранения результата умножения
            for t in range(mid):  # проходим по всем строкам матрицы B
                s ^= (
                    A[i][t] & B[t][j]
                )  # выполняем побитовое умножение и сложение по модулю 2
            C[i][j] = s  # записываем результат в матрицу C
    return C  # возвращаем матрицу C


def transpose(M):  # Транспонирование матрицы
    return [list(col) for col in zip(*M)]


def rref_mod2(M):  # Приведение к ступенчатому виду (RREF) над GF(2)
    A = [row[:] for row in M]  # создаем копию матрицы M
    n_rows = len(A)  # количество строк в матрице A
    n_cols = len(A[0]) if n_rows else 0  # количество столбцов в матрице A
    pivot_cols = (
        []
    )  # список для хранения индексов столбцов, содержащих ведущие элементы
    r = 0  # индекс строки, в которой будет найден ведущий элемент
    for c in range(n_cols):  # проходим по всем столбцам матрицы A
        piv = None  # переменная для хранения индекса строки, содержащей ведущий элемент
        for i in range(
            r, n_rows
        ):  # проходим по всем строкам матрицы A, начиная с текущей строки r
            if A[i][c]:  # если элемент в текущей позиции является ведущим элементом
                piv = i  # сохраняем индекс строки, содержащей ведущий элемент
                break
        if piv is None:
            continue  # если ведущий элемент не найден, переходим к следующему столбцу
        A[r], A[piv] = A[piv], A[r]  # меняем местами строки r и piv
        for i in range(n_rows):  # проходим по всем строкам матрицы A
            if (
                i != r and A[i][c]
            ):  # если текущая строка не равна строке r и элемент в текущей позиции является ведущим элементом
                A[i] = [
                    x ^ y for x, y in zip(A[i], A[r])
                ]  # выполняем побитовое сложение по модулю 2
        pivot_cols.append(c)  # добавляем индекс столбца c в список pivot_cols
        r += 1  # увеличиваем индекс строки r
        if r == n_rows:  # если достигли последней строки
            break  # выходим из цикла
    return A, pivot_cols  # возвращаем матрицу A и список pivot_cols


def nullspace_basis_mod2(A):  # Базис решений A*x=0 над GF(2), как список векторов-строк
    R, piv = rref_mod2(A)  # приводим матрицу A к ступенчатому виду
    n = len(A[0])  # количество столбцов в матрице A
    free = [j for j in range(n) if j not in piv]  # список свободных столбцов
    basis = []  # список для хранения базисных векторов
    for f in free:  # проходим по всем свободным столбцам
        x = [0] * n  # создаем вектор x с нулевыми значениями
        x[f] = 1  # устанавливаем значение 1 в свободном столбце
        # Из RREF: x[piv_i] = sum(R[i][j] * x[j] for j in free)
        for i, pc in enumerate(piv):  # проходим по всем ведущим столбцам
            s = 0  # инициализируем переменную s для хранения результата суммы
            for j in free:  # проходим по всем свободным столбцам
                s ^= R[i][j] & x[j]  # выполняем побитовое сложение по модулю 2
            x[pc] = s  # записываем результат в вектор x
        basis.append(x)  # добавляем вектор x в список базисных векторов
    return basis  # возвращаем список базисных векторов


def print_matrix(title, M):
    print(title)
    for row in M:
        print(" ", "".join(map(str, row)))

In [10]:
# Параметры BCH(15,7): длина кода n=15, размерность k=7, избыточность r
n = 15
k = 7
r = n - k

# Минимальные образующие многочлены с соответствующими корнями для BCH-кода (из конспекта)
M1 = [1, 1, 0, 0, 1]  # Многочлен 1 + x + x^4 (корень alpha^1)
M3 = [1, 1, 1, 1, 1]  # Многочлен 1 + x + x^2 + x^3 + x^4 (корень alpha^3)

# Перемножаем минимальные многочлены, чтобы получить порождающий многочлен кода g(x)
g = poly_mul_bin(M1, M3)  # результатом будет многочлен степени 8
print("M1(x) =", poly_to_str(M1))
print("M3(x) =", poly_to_str(M3))
print("g(x) = M1(x)*M3(x) =", poly_to_str(g))
print("deg g =", len(g) - 1)  # Степень порождающего многочлена

# Строим многочлен x^15 + 1 в представлении над GF(2) (также x^15 - 1, так как -1 = 1 в GF(2))
x15_plus_1 = [1] + [0] * 14 + [1]
# Делим x^15 + 1 на g(x), чтобы получить проверочный многочлен h(x) и остаток rem
h, rem = poly_div_rem_bin(x15_plus_1, g)
print("\nh(x) = (x^15 + 1)/g(x) =", poly_to_str(h))
print("Остаток от деления x^15+1 на g(x):", rem)

# Строим порождающую матрицу G размерности (k x n) для циклического кода с порождающим многочленом g(x)
G = cyclic_generator_matrix(g, n=n, k=k)

# Строим проверочную матрицу H как базис пространства решений уравнения G*x^T=0,
# то есть ортогонального дополнения к строкам G. H — (n-k) x n, здесь 8 x 15.
H = nullspace_basis_mod2(G)

print_matrix("\nПорождающая матрица G (7x15):", G)
print_matrix("\nПроверочная матрица H (8x15):", H)

# Проверяем ортогональность породжающей и проверочной матриц: должна получиться нулевая матрица
P = mat_mul_mod2(G, transpose(H))
print_matrix("\nПроверка G * H^T (должна быть нулевая матрица 7x8):", P)

# Проверяем, является ли результат действительно нулевой матрицей
is_zero = all(all(v == 0 for v in row) for row in P)
print(
    "\nРезультат проверки:", "G * H^T = 0 (верно)" if is_zero else "НЕ нулевая матрица"
)

M1(x) = 1 + x + x^4
M3(x) = 1 + x + x^2 + x^3 + x^4
g(x) = M1(x)*M3(x) = 1 + x^4 + x^6 + x^7 + x^8
deg g = 8

h(x) = (x^15 + 1)/g(x) = 1 + x^4 + x^6 + x^7
Остаток от деления x^15+1 на g(x): [0, 0, 0, 0, 0, 0, 0, 0]

Порождающая матрица G (7x15):
  100010111000000
  010001011100000
  001000101110000
  000100010111000
  000010001011100
  000001000101110
  000000100010111

Проверочная матрица H (8x15):
  110100010000000
  011010001000000
  001101000100000
  000110100010000
  110111000001000
  011011100000100
  111001100000010
  101000100000001

Проверка G * H^T (должна быть нулевая матрица 7x8):
  00000000
  00000000
  00000000
  00000000
  00000000
  00000000
  00000000

Результат проверки: G * H^T = 0 (верно)
